In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.feature import PCA as SparkPCA
import yaml


In [0]:


import yaml




CONFIG_PATH = "/Volumes/bmqg/default_bronze/fatemeh/config_mixed.yaml"

with open(CONFIG_PATH, "r") as f:
    CONFIG = yaml.safe_load(f)


gwas_table = CONFIG["data"]["gwas_table_newharvested"]

TAGLO_TABLE =CONFIG["paths"]["TAGLO_TABLE"]
AROMA_MATRIX_PATH = CONFIG["paths"]["aroma_matrix_newharvested"]
df_pheno = pd.read_csv(AROMA_MATRIX_PATH)



In [0]:
# PHENO_INPUT = CONFIG["data"]["aroma_data"]

# pheno_df = (
#     pd.read_csv(PHENO_INPUT)
#     .set_index("Variety")
# )

# pheno_df.head()


In [0]:
TAGLO_TABLE = CONFIG["paths"]["TAGLO_TABLE"]
df_taglo  = spark.table(TAGLO_TABLE)

display(df_taglo.limit(10))
print("Rows:", df_taglo.count())


taglo_id,trait,chrom,start,end,position,haplotype_count,num_genotypes_called,allele_freq,variety,ploidy,value
314043,,ST4.03ch07,8950000,9000000,8960017,21,840,0.5338983050847458,ALOUETTE,4,2.0
314043,,ST4.03ch07,8950000,9000000,8960017,21,840,0.5338983050847458,ALPINE RUSSET,4,3.0
314043,,ST4.03ch07,8950000,9000000,8960017,21,840,0.5338983050847458,ALTHEA,4,2.0
314043,,ST4.03ch07,8950000,9000000,8960017,21,840,0.5338983050847458,ALTURAS,4,2.0
314043,,ST4.03ch07,8950000,9000000,8960017,21,840,0.5338983050847458,ALTUS,4,3.0
314043,,ST4.03ch07,8950000,9000000,8960017,21,840,0.5338983050847458,ALVERSTONE R.,4,2.0
314043,,ST4.03ch07,8950000,9000000,8960017,21,840,0.5338983050847458,AM 66- 42,4,2.0
314043,,ST4.03ch07,8950000,9000000,8960017,21,840,0.5338983050847458,AM 66- 148,4,2.0
314043,,ST4.03ch07,8950000,9000000,8960017,21,840,0.5338983050847458,AM 78-3704,4,2.0
314043,,ST4.03ch07,8950000,9000000,8960017,21,840,0.5338983050847458,AM 78-3736,4,2.0


Rows: 58052698


In [0]:
import pyspark.sql.functions as F


df_taglo_sdf = spark.table(CONFIG["paths"]["TAGLO_TABLE"])


df_pheno_sdf = (
    spark.read.option("header", True).option("inferSchema", True)
    .csv(CONFIG["paths"]["aroma_matrix"])
)

# (normalize Variety)
taglo = df_taglo_sdf.withColumn(
    "variety_key", F.upper(F.trim(F.col("variety").cast("string")))
)

pheno = df_pheno_sdf.withColumn(
    "variety_key", F.upper(F.trim(F.col("Variety").cast("string")))
)

merged = taglo.join(pheno, on="variety_key", how="inner")

display(merged.limit(10))
print("Merged rows:", merged.count())


variety_key,taglo_id,trait,chrom,start,end,position,haplotype_count,num_genotypes_called,allele_freq,variety,ploidy,value,Variety,(E)-2-Decenal,(E)-2-Heptenal,"(E, E)-2,4-Decadienal","(E, E)-3,5-Octadien-2-one",1-Heptanol,"1-Hexanol, 2-ethyl-",1-Nonanol,1-Octen-3-ol,1-Octyn-3-ol,1-Penten-3-one,1-Phenylethanol,"2(3H)-Furanone, dihydro-5-pentyl-","2,3-Butanedione",2-Decanone,2-Ethylfuran,2-Heptanone,2-Methylbutanal,2-Methylpropanal,2-Nonanone,"2-Nonenal, (E)-","2-Octenal, (E)-","2-Propanone, 1-methoxy-",2-Tridecenal,3-Carene,3-Furaldehyde,3-Heptanone,3-Methylbutanal,6-Methyl-5-hepten-2-one,"Acetic acid, ethenyl ester","Acetic acid, methyl ester",Benzaldehyde,Benzoic acid,"Benzoic acid, methyl ester",Benzyl alcohol,"Butanal, 3-methyl-",Butanoic acid,Decanal,Dimethyl disulfide,Dimethyl phthalate,Dimethyl sulfide,Dimethyl trisulfide,Dodecanal,Ethanol,"Ethanol, 2-phenoxy-",Ethyl butanoate,"Furan, 2-pentyl-",Furfural,Heptanal,Hexanal,Hexanoic acid,Linalool,Methional,Methyl 8-oxooctanoate,Nonanal,"Nonanoic acid, 9-oxo-, methyl ester","Octanoic acid, methyl ester",Pentanal,Phenylacetaldehyde,Propanal,Styrene,Tetradecane,Tridecanal,Tridecane,Undecanal,alpha-Terpineol,n-Hexadecanoic acid,n-Hexane
ALOUETTE,314043,,ST4.03ch07,8950000,9000000,8960017,21,840,0.5338983050847458,ALOUETTE,4,2.0,ALOUETTE,195913.9870211408,105425.83310938108,116490.01807281972,678804.0641240367,90006.9274835497,1533476.9022452713,70176.3698514762,75910.62131806751,353220.03435519943,353135.1644814982,106740.52543863998,681068.6664682614,2258747.431507581,275482.3928973752,56449.59027894123,35911.60798533387,363214.6480531743,457749.14451480994,275107.0677416283,195913.9870211408,268847.7827706501,5962349.585717665,195913.9870211408,106416.33306394304,661678.2724668736,363214.6480531743,2900663.6205276856,293606.59164019016,2258747.431507581,241135.39686789503,1738506.7028450856,533597.5087841562,462455.4689396789,318145.05512335093,86456.62562406348,680869.933906862,3784646.0850318833,1310860.0606746986,1051764.987674809,502062.8157124129,437522.17604408256,1497252.753294984,18516.446729032625,246592.89874633032,124424.56055076887,506654.750307511,661678.2724668736,562616.6798850796,1610618.873160969,135414.94562403226,64238.11440275864,89027.49751251414,143645.2148047539,1.283951306720216E7,456226.7582089155,536635.428548999,516918.7336046246,736163.4473362742,17467.434004403418,495876.80348953657,1681311.260817449,939639.7602479096,438183.2122032206,957462.9608672236,96496.57354028852,1386045.2802166145,116435.19529726105
ALTHEA,314043,,ST4.03ch07,8950000,9000000,8960017,21,840,0.5338983050847458,ALTHEA,4,2.0,ALTHEA,272957.9012112527,176872.29204490432,1184138.4742814123,1585794.888334833,346863.538669355,4079950.144624443,341051.62233013293,179545.42285865315,302785.6183470979,1172902.8495582463,331039.0718550213,4490904.553230276,3989266.812035223,1513203.4103037536,130293.18244994378,253024.8596130324,359652.48230014567,508311.7748451286,1151398.9539377335,272957.9012112527,825559.5048411654,8209084.363371965,272957.9012112527,161372.71193843722,974873.9287535856,359652.48230014567,2014014.0785903244,592943.5445280041,3989266.812035223,90429.04759957804,4276341.967640292,1944281.311336556,544680.0897519318,880738.1269209541,43515.309566380114,1961338.1070427955,7189689.23973729,1958330.347790812,3662228.643016526,707057.1999185616,489259.8727589474,4717462.618512353,33449.179574228656,1993005.5499591704,294905.0455636358,1117441.7172977056,974873.9287535856,1110492.453937692,2513841.3191959634,270620.3730087784,107593.48942641546,365573.50601952,1836073.0928967795,2.369050393608192E7,5005056.521259663,1421674.4955183496,1727448.9025701708,1080912.6638397842,102849.26673583404,1698228.5823229454,1.0436393654258003E7,4423660.310999733,1265574.9564026648,3094299.887907712,373942.6265810778,2856549.0615161,106444.45232917914
ALTURAS,314043,,ST4.03ch07,8950000,9000000,8960017,21,840,0.5338983050847458,ALTURAS,4,2.0,ALTURAS,149731.38059251895,157937.3290595055,173

Merged rows: 7267386


In [0]:


exclude = {
    "variety_key", "Variety", "variety",
    "taglo_id", "trait", "chrom", "start", "end", "position",
    "haplotype_count", "num_genotypes_called", "allele_freq", "ploidy"
}

num_cols = []
for name, dtype in merged.dtypes:
    if name in exclude:
        continue
    if dtype in ("int", "bigint", "double", "float", "smallint", "tinyint", "decimal"):
        num_cols.append(name)

print("Numeric columns:", len(num_cols))
print("Example:", num_cols[:15])

# Variety
merged_uniq = (
    merged
    .groupBy("variety_key")
    .agg(*[F.avg(F.col(c)).alias(c) for c in num_cols])
)

print("Rows before:", merged.count())
print("Rows after uniq:", merged_uniq.count())


Numeric columns: 68
Example: ['value', '(E)-2-Decenal', '(E)-2-Heptenal', '(E, E)-2,4-Decadienal', '(E, E)-3,5-Octadien-2-one', '1-Heptanol', '1-Hexanol, 2-ethyl-', '1-Nonanol', '1-Octen-3-ol', '1-Octyn-3-ol', '1-Penten-3-one', '1-Phenylethanol', '2(3H)-Furanone, dihydro-5-pentyl-', '2,3-Butanedione', '2-Decanone']
Rows before: 7267386
Rows after uniq: 94


In [0]:

for _name in ["model", "pipe", "assembler", "scaler", "pca", "pca_sdf", "pca_base", "merged_uniq", "merged_small"]:
    if _name in globals():
        del globals()[_name]

# ---------- 1) PICK NUMERIC COLUMNS ----------
exclude = {
    "variety_key", "Variety", "variety",
    "taglo_id", "trait", "chrom", "start", "end", "position",
    "haplotype_count", "num_genotypes_called", "allele_freq", "ploidy"
}

num_cols = [c for c, t in merged.dtypes
            if c not in exclude and t in ("int","bigint","double","float","smallint","tinyint","decimal")]

print("Numeric columns (raw):", len(num_cols))
print("Example:", num_cols[:15])

if len(num_cols) == 0:
    raise ValueError("No numeric columns found. Phenotypes may be strings; cast them to double first.")

# ---------- 2) REDUCE INPUT (ONLY NEEDED COLUMNS) ----------
# This dramatically reduces what Spark has to move through the pipeline.
merged_small = merged.select(["variety_key"] + num_cols)

# ---------- 3) UNIQUE VARIETIES (MEAN PER VARIETY) ----------
merged_uniq = (
    merged_small
    .groupBy("variety_key")
    .agg(*[F.avg(F.col(c)).alias(c) for c in num_cols])
)

print("Rows (merged_small):", merged_small.count())
print("Rows (merged_uniq / unique varieties):", merged_uniq.count())

# ---------- 4) PCA PIPELINE ----------
assembler = VectorAssembler(inputCols=num_cols, outputCol="features_raw_v2", handleInvalid="skip")
scaler    = StandardScaler(inputCol="features_raw_v2", outputCol="features_v2", withMean=True, withStd=True)
pca       = SparkPCA(k=2, inputCol="features_v2", outputCol="pca_vec_v2")

pipe  = Pipeline(stages=[assembler, scaler, pca])
model = pipe.fit(merged_uniq)

pca_base = model.transform(merged_uniq).select("variety_key", "pca_vec_v2")

print("Rows after PCA:", pca_base.count())

# ---------- 5) COLLECT + EXTRACT PCs IN PYTHON (SAFE) ----------
rows = pca_base.collect()  
varieties = [r["variety_key"] for r in rows]
pc1 = np.array([float(r["pca_vec_v2"].toArray()[0]) for r in rows])
pc2 = np.array([float(r["pca_vec_v2"].toArray()[1]) for r in rows])

dfp = pd.DataFrame({"variety": varieties, "PC1": pc1, "PC2": pc2}).dropna()
print("dfp shape:", dfp.shape)

# ---------- 6) ROBUST OUTLIERS (MAD DISTANCE) ----------
def robust_z(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    if mad == 0 or np.isnan(mad):
        s = np.std(x)
        return (x - np.mean(x)) / (s if s != 0 else 1.0)
    return 0.6745 * (x - med) / mad

dfp["rz1"] = robust_z(dfp["PC1"].values)
dfp["rz2"] = robust_z(dfp["PC2"].values)
dfp["r_dist"] = np.sqrt(dfp["rz1"]**2 + dfp["rz2"]**2)

THR = 3.5
dfp["is_outlier"] = dfp["r_dist"] > THR
outliers = dfp[dfp["is_outlier"]].sort_values("r_dist", ascending=False)

print("Outliers found:", len(outliers))
display(outliers[["variety","PC1","PC2","r_dist"]])

# ---------- 7) PLOT + LABEL OUTLIERS ----------
topN = 30
rng = np.random.default_rng(42)

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(dfp["PC1"], dfp["PC2"], alpha=0.6)

if len(outliers) > 0:
    ax.scatter(outliers["PC1"], outliers["PC2"], alpha=0.9)
    for _, r in outliers.head(topN).iterrows():
        dx, dy = rng.integers(-25, 26), rng.integers(-25, 26)
        ax.annotate(
            r["variety"],
            (r["PC1"], r["PC2"]),
            xytext=(dx, dy),
            textcoords="offset points",
            fontsize=9,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8),
            arrowprops=dict(arrowstyle="-", alpha=0.5),
        )

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title(f"PCA: PC1 vs PC2 (Outliers labeled, thr={THR})")
plt.tight_layout()
plt.show()

print("\nOutlier varieties:")
for v in outliers["variety"].tolist():
    print("-", v)


Numeric columns (raw): 68
Example: ['value', '(E)-2-Decenal', '(E)-2-Heptenal', '(E, E)-2,4-Decadienal', '(E, E)-3,5-Octadien-2-one', '1-Heptanol', '1-Hexanol, 2-ethyl-', '1-Nonanol', '1-Octen-3-ol', '1-Octyn-3-ol', '1-Penten-3-one', '1-Phenylethanol', '2(3H)-Furanone, dihydro-5-pentyl-', '2,3-Butanedione', '2-Decanone']
Rows (merged_small): 7267386
Rows (merged_uniq / unique varieties): 94


In [0]:
# ============================================================
# COMPLETE: PCA on merged_uniq (pandas) + robust outliers
# + PC loadings (top +/-)
# + per-outlier feature contributions (what drives separation)
# + l log1p for VOC skew
# ============================================================



# ------------------------------------------------------------
# 0) Spark -> pandas, index by variety
# ------------------------------------------------------------
df = merged_uniq.toPandas()

if "variety_key" not in df.columns:
    raise ValueError("merged_uniq must contain 'variety_key' column.")

df["variety_key"] = df["variety_key"].astype(str).str.strip().str.upper()
df = df.set_index("variety_key")

# ------------------------------------------------------------
# 1) CONFIG
# ------------------------------------------------------------
TARGET_GROUP = ["ALTHEA", "DONALD", "VIOLET QUEEN", "KENNEBEC", "HANSA"]   # highlight these
N_COMPONENTS = 5
FOCUS_PC = "PC2"          # "PC1" or "PC2"
TOPK_LOADINGS = 15
TOPK_CONTRIB = 15
OUTLIER_THR = 3.5         # robust distance threshold (MAD-based)
USE_LOG1P = True          # recommended for VOC data

# drop columns that should not be in phenotype PCA, if present
DROP_COLS = {"value"}     # add more if needed

# ------------------------------------------------------------
# 2) Numeric matrix X
# ------------------------------------------------------------
X = df.select_dtypes(include=[np.number]).copy()

for c in list(DROP_COLS):
    if c in X.columns:
        X = X.drop(columns=[c])

if X.shape[1] < 2:
    raise ValueError("Need at least 2 numeric features for PCA.")

# optional: log transform to reduce skew (VOC-friendly)
if USE_LOG1P:
    X = np.log1p(X)

feature_names = X.columns.tolist()

# ------------------------------------------------------------
# 3) Scale + PCA
# ------------------------------------------------------------
scaler = StandardScaler()
Z = scaler.fit_transform(X.values)   # standardized

pca = PCA(n_components=min(N_COMPONENTS, Z.shape[0]-1, Z.shape[1]), random_state=42)
pcs = pca.fit_transform(Z)

# scores (PC1/PC2)
scores = pd.DataFrame(
    pcs[:, :2],
    columns=["PC1", "PC2"],
    index=df.index
)

# loadings (feature weights on PCs)
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)],
    index=feature_names
)

evr = pca.explained_variance_ratio_
print("Explained variance ratio:", evr[:5], " | sum(PC1..PC2) =", evr[:2].sum())

# ------------------------------------------------------------
# 4) Robust outliers in PCA space (MAD distance)
# ------------------------------------------------------------
def robust_z(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    if mad == 0 or np.isnan(mad):
        s = np.std(x)
        return (x - np.mean(x)) / (s if s != 0 else 1.0)
    return 0.6745 * (x - med) / mad

scores["rz1"] = robust_z(scores["PC1"].values)
scores["rz2"] = robust_z(scores["PC2"].values)
scores["r_dist"] = np.sqrt(scores["rz1"]**2 + scores["rz2"]**2)
scores["is_outlier"] = scores["r_dist"] > OUTLIER_THR

outliers = scores[scores["is_outlier"]].sort_values("r_dist", ascending=False)
print("\nOutliers found:", len(outliers))
display(outliers[["PC1","PC2","r_dist"]])

# ------------------------------------------------------------
# 5) Top +/- loadings for a chosen PC (global axis meaning)
# ------------------------------------------------------------
if FOCUS_PC not in loadings.columns:
    raise ValueError(f"{FOCUS_PC} not available. Available: {list(loadings.columns)}")

print("\n====================================================")
print(f" TOP + LOADINGS on {FOCUS_PC} (high along {FOCUS_PC})")
print("====================================================")
display(loadings[FOCUS_PC].sort_values(ascending=False).head(TOPK_LOADINGS))

print("\n====================================================")
print(f" TOP - LOADINGS on {FOCUS_PC} (low along {FOCUS_PC})")
print("====================================================")
display(loadings[FOCUS_PC].sort_values(ascending=True).head(TOPK_LOADINGS))

# ------------------------------------------------------------
# 6) Mean difference: target group vs rest (phenotype shift)
# ------------------------------------------------------------
tg = [t.strip().upper() for t in TARGET_GROUP]
present = [t for t in tg if t in df.index]
missing = [t for t in tg if t not in df.index]
if missing:
    print("\n[WARN] Missing target varieties (ignored):", missing)
if len(present) == 0:
    raise ValueError("None of TARGET_GROUP varieties exist in data index.")

group_mean = df.loc[present, feature_names].mean(numeric_only=True)
others_mean = df.drop(index=present, errors="ignore")[feature_names].mean(numeric_only=True)
difference = (group_mean - others_mean).sort_values(ascending=False)

print("\n====================================================")
print(" HIGHER in TARGET GROUP (mean difference)")
print("====================================================")
display(difference.head(20))

print("\n====================================================")
print(" LOWER in TARGET GROUP (mean difference)")
print("====================================================")
display(difference.tail(20))

# ------------------------------------------------------------
# 7) Per-outlier feature contributions (BEST explanation)
# PC score = sum(z_i * loading_i)
# so contribution_i = z_i * loading_i
# ------------------------------------------------------------
pc_to_use = FOCUS_PC  # "PC1" or "PC2"
pc_idx = int(pc_to_use.replace("PC","")) - 1  # 0 for PC1, 1 for PC2

contrib = Z * loadings.iloc[:, pc_idx].values  # shape (n_samples, n_features)
contrib_df = pd.DataFrame(contrib, index=df.index, columns=feature_names)

# pick which varieties to explain:
EXPLAIN_VAR = []
EXPLAIN_VAR += present
EXPLAIN_VAR += outliers.index.tolist()  # include detected outliers
EXPLAIN_VAR = list(dict.fromkeys(EXPLAIN_VAR))  # unique keep order

print("\n==============================")
print(f"Top feature contributions to {pc_to_use} for target/outlier varieties")
print("==============================")
for v in EXPLAIN_VAR:
    if v not in contrib_df.index:
        continue
    s = contrib_df.loc[v].sort_values(key=lambda x: np.abs(x), ascending=False).head(TOPK_CONTRIB)
    print(f"\n--- {v} (score {pc_to_use} = {scores.loc[v, pc_to_use]:.3f}) ---")
    display(pd.DataFrame({"feature": s.index, f"{pc_to_use}_contribution": s.values}))

# ------------------------------------------------------------
# 8) Plot PCA scores + label targets + label outliers
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(scores["PC1"], scores["PC2"], alpha=0.6)

# highlight target group
if present:
    ax.scatter(scores.loc[present, "PC1"], scores.loc[present, "PC2"], alpha=0.95)
    for v in present:
        ax.annotate(v, (scores.loc[v, "PC1"], scores.loc[v, "PC2"]),
                    xytext=(5,5), textcoords="offset points", fontsize=9)

# highlight outliers
if len(outliers) > 0:
    ax.scatter(outliers["PC1"], outliers["PC2"], alpha=0.95)
    for v in outliers.index.tolist():
        ax.annotate(v, (scores.loc[v, "PC1"], scores.loc[v, "PC2"]),
                    xytext=(5,-10), textcoords="offset points", fontsize=9)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title(f"PCA: PC1 vs PC2 (merged_uniq) | outliers thr={OUTLIER_THR} | log1p={USE_LOG1P}")
plt.tight_layout()
plt.show()

print("\nOutlier varieties:")
for v in outliers.index.tolist():
    print("-", v)


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:134)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:465)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.data

# Genotype PCA Outlier Ranking and Phenotype Association Analysis

In [0]:
# ============================================================
# COMPLETE: Genotype PCA ranking (top genetically different)
#          + check effect on phenotype (merged_uniq)
#          + correlation of genotype PCs with each phenotype trait
#

# ----------------------------
# 1) Robust distance in genotype PCA space
# ----------------------------
def robust_z(x):
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    if mad == 0 or np.isnan(mad):
        s = np.std(x)
        return (x - np.mean(x)) / (s if s != 0 else 1.0)
    return 0.6745 * (x - med) / mad

# normalize variety names
g = gdf.copy()
g["variety"] = g["variety"].astype(str).str.strip().str.upper()

# robust z + distance
g["rz1"] = robust_z(g["PC1"].values)
g["rz2"] = robust_z(g["PC2"].values)
g["g_dist"] = np.sqrt(g["rz1"]**2 + g["rz2"]**2)

# top-k genetically different varieties (by distance)
TOPK_DIFF = 10
top_genetic = g.sort_values("g_dist", ascending=False).head(TOPK_DIFF)

print("====================================================")
print("TOP genetically different varieties (Genotype PCA distance)")
print("====================================================")
display(top_genetic[["variety", "PC1", "PC2", "g_dist"]])

print("\nTop genetic variety names:")
for v in top_genetic["variety"].tolist():
    print("-", v)

# visualize distances (bar plot)
plt.figure(figsize=(8,3))
plt.bar(top_genetic["variety"], top_genetic["g_dist"])
plt.xticks(rotation=60, ha="right")
plt.ylabel("Genotype PCA robust distance")
plt.title(f"Top {TOPK_DIFF} genetically different varieties")
plt.tight_layout()
plt.show()

# ----------------------------
# 2) Load phenotype (merged_uniq) from Spark -> pandas
# ----------------------------
ph = merged_uniq.toPandas()
ph["variety_key"] = ph["variety_key"].astype(str).str.strip().str.upper()
ph = ph.set_index("variety_key")

# numeric phenotype columns only
ph_num = ph.select_dtypes(include=[np.number]).copy()
print("\nPhenotype matrix shape:", ph_num.shape)

# ----------------------------
# 3) Compare phenotype in top genetic group vs the rest
# ----------------------------
group = [v for v in top_genetic["variety"].tolist() if v in ph_num.index]
missing = [v for v in top_genetic["variety"].tolist() if v not in ph_num.index]

print("\nGroup present in phenotype:", len(group), "/", TOPK_DIFF)
if missing:
    print("[WARN] These were missing in phenotype index:", missing)

if len(group) >= 2:
    group_mean = ph_num.loc[group].mean()
    rest_mean  = ph_num.drop(index=group, errors="ignore").mean()

    diff = (group_mean - rest_mean).sort_values(ascending=False)

    print("\n====================================================")
    print("Traits HIGHER in TOP genetic-different group (mean difference)")
    print("====================================================")
    display(diff.head(20))

    print("\n====================================================")
    print("Traits LOWER in TOP genetic-different group (mean difference)")
    print("====================================================")
    display(diff.tail(20))
else:
    print("Not enough varieties in group to compare phenotype means reliably.")

# choose a trait to visualize group vs rest
# (replace with any trait name from ph_num.columns)
trait_to_plot = ph_num.columns[0]

if trait_to_plot in ph_num.columns and len(group) >= 2:
    plt.figure(figsize=(6,4))
    plt.boxplot(
        [ph_num.loc[group, trait_to_plot].dropna().values,
         ph_num.drop(index=group, errors="ignore")[trait_to_plot].dropna().values],
        labels=["Top genetic", "Rest"]
    )
    plt.ylabel(trait_to_plot)
    plt.title(f"Phenotype difference for: {trait_to_plot}")
    plt.tight_layout()
    plt.show()

# ----------------------------
# 4) Correlation: genotype PCs vs each phenotype trait
# ----------------------------
# Join genotype PCs with phenotype traits
gp = g.set_index("variety")[["PC1", "PC2"]].join(ph_num, how="inner")
print("\nJoined genotype+phenotype shape:", gp.shape)

# correlations
corr_pc1 = gp.corr(numeric_only=True)["PC1"].drop("PC1").sort_values(ascending=False)
corr_pc2 = gp.corr(numeric_only=True)["PC2"].drop("PC2").sort_values(ascending=False)

print("\n====================================================")
print("Top phenotype traits correlated with Genotype PC1")
print("====================================================")
display(corr_pc1.head(15))

print("\n====================================================")
print("Bottom phenotype traits correlated with Genotype PC1")
print("====================================================")
display(corr_pc1.tail(15))

print("\n====================================================")
print("Top phenotype traits correlated with Genotype PC2")
print("====================================================")
display(corr_pc2.head(15))

print("\n====================================================")
print("Bottom phenotype traits correlated with Genotype PC2")
print("====================================================")
display(corr_pc2.tail(15))

#  scatter plot for the strongest correlated trait (by abs correlation)
best_trait_pc1 = corr_pc1.abs().sort_values(ascending=False).index[0]
best_trait_pc2 = corr_pc2.abs().sort_values(ascending=False).index[0]

plt.figure(figsize=(6,4))
plt.scatter(gp["PC1"], gp[best_trait_pc1], alpha=0.7)
plt.xlabel("Genotype PC1")
plt.ylabel(best_trait_pc1)
plt.title(f"Genotype PC1 vs phenotype: {best_trait_pc1}")
plt.tight_layout()
plt.show()




com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:134)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:465)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.data